# Concrete Compressive Strength Prediction with PyTorch

**Feedforward ANN regression · engineering-informed features · group-aware cross-validation · SHAP interpretation**

This notebook reproduces the development workflow and experiments behind a neural network trained on the concrete compressive strength dataset by Yeh (1998). It includes experimental data cleaning, feature engineering, a baseline model, 5-fold cross-validation, hyperparameter optimization, early stopping, and final held-out evaluation.

**Dataset:** [UCI Concrete Compressive Strength](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength), CC BY 4.0 (attribution in repository README).

**Run:** From the repository root (`jupyter lab`) or the `notebooks/` directory. Install dependencies first with `pip install -r requirements.txt`. In Colab, upload the `data` directory or clone the repository and set the working directory to its root. **All results are reference results from the original experiment; runtime/version differences may produce small changes.**

## 1. Imports and data loading

All data paths are relative to the project. No personal Google Drive mount is required.

In [ ]:
import copy
import shap
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.preprocessing import StandardScaler


from pathlib import Path

# Supports running the notebook from the repository root or notebooks/ folder.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA_DIR = next((directory for directory in candidates
                 if (directory / 'train-val.csv').exists()
                 and (directory / 'test.csv').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Required files data/train-val.csv and data/test.csv not found. See README.')

train_df = pd.read_csv(DATA_DIR / 'train-val.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
print('Training/validation:', train_df.shape)
print('Test:', test_df.shape)
train_df.head()


### Dataset schema

In [ ]:
train_df.info()

### Exact duplicate removal

In [ ]:
# Check exact duplicates
print("Duplicates:", train_df.duplicated().sum())

# Remove exact duplicate rows
train_df = train_df.drop_duplicates().reset_index(drop=True)

# Confirm the new dataset size
print("Cleaned dataset shape:", train_df.shape)


### Repeated inputs and experimental variability

In [ ]:

# Select the eight input columns
feature_cols = train_df.columns[:-1]

# Find observations having identical inputs
same_inputs = train_df.duplicated(
    subset=feature_cols,
    keep=False
)

# Display matching observations
print("Rows with repeated inputs:", same_inputs.sum())

train_df.loc[same_inputs].sort_values(
    by=list(feature_cols)
)


## 2. Group-aware train/validation split and standardization

In [ ]:

# Separate inputs and output
X = train_df.iloc[:, :-1]
y = train_df.iloc[:, -1]

# Give identical input mixtures the same group ID
groups = pd.factorize(pd.MultiIndex.from_frame(X))[0]

# Split without separating identical mixtures
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

# Standardize using training data only
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Training:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)


## 3. Engineering-informed features

New inputs: total binder, water-to-binder ratio, SCM fraction and logarithmic curing age. The original eight input columns are retained.

In [ ]:

# Create a copy of the original input features
X_engineered = X.copy()

# Extract the relevant features
cement = X.iloc[:, 0]
slag = X.iloc[:, 1]
fly_ash = X.iloc[:, 2]
water = X.iloc[:, 3]
age = X.iloc[:, 7]

# Engineer new features
binder = cement + slag + fly_ash

X_engineered["Total_Binder"] = binder

X_engineered["Water_Binder_Ratio"] = water / binder

X_engineered["SCM_Fraction"] = (slag + fly_ash) / binder

X_engineered["Log_Age"] = np.log1p(age)

print(X_engineered.shape)
X_engineered.head()


## 4. PyTorch tensors and baseline model

In [ ]:

# Convert input features into tensors
X_train_tensor = torch.tensor(
    X_train_scaled, dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val_scaled, dtype=torch.float32
)

# Convert target values into tensors
y_train_tensor = torch.tensor(
    y_train.to_numpy(), dtype=torch.float32
).reshape(-1, 1)

y_val_tensor = torch.tensor(
    y_val.to_numpy(), dtype=torch.float32
).reshape(-1, 1)

# Check tensor dimensions
print("X_train:", X_train_tensor.shape)
print("y_train:", y_train_tensor.shape)

print("X_val:", X_val_tensor.shape)
print("y_val:", y_val_tensor.shape)


In [ ]:

# Use the same training and validation indices
X_train_eng = X_engineered.iloc[train_idx]
X_val_eng = X_engineered.iloc[val_idx]

# Standardization
scaler_eng = StandardScaler()

X_train_eng_scaled = scaler_eng.fit_transform(X_train_eng)
X_val_eng_scaled = scaler_eng.transform(X_val_eng)

# Convert to PyTorch tensors
X_train_eng_tensor = torch.tensor(
    X_train_eng_scaled, dtype=torch.float32
)

X_val_eng_tensor = torch.tensor(
    X_val_eng_scaled, dtype=torch.float32
)

print(X_train_eng_tensor.shape)
print(X_val_eng_tensor.shape)


### Define a configurable feedforward ANN

In [ ]:

class ConcreteANN(nn.Module):
    def __init__(self, input_dim, hidden1=32, hidden2=16):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),

            nn.Linear(hidden1, hidden2),
            nn.ReLU(),

            nn.Linear(hidden2, 1)
        )

    def forward(self, x):
        return self.network(x)


# Set seed for reproducibility
torch.manual_seed(42)

# Baseline ANN
baseline_model = ConcreteANN(input_dim=8)

print(baseline_model)


### Baseline learning curves (first 500 epochs)

In [ ]:

# Loss function and optimizer
criterion = nn.MSELoss()

optimizer = optim.Adam(
    baseline_model.parameters(),
    lr=0.001
)

epochs = 500

train_losses = []
val_losses = []

for epoch in range(epochs):

    # Training mode
    baseline_model.train()

    # 1. Forward propagation
    predictions = baseline_model(X_train_tensor)

    # 2. Calculate training loss
    loss = criterion(predictions, y_train_tensor)

    # 3. Reset gradients
    optimizer.zero_grad()

    # 4. Backpropagation
    loss.backward()

    # 5. Update weights
    optimizer.step()

    # Validation
    baseline_model.eval()

    with torch.no_grad():
        val_predictions = baseline_model(X_val_tensor)
        val_loss = criterion(
            val_predictions, y_val_tensor
        )

    # Save RMSE values
    train_losses.append(np.sqrt(loss.item()))
    val_losses.append(np.sqrt(val_loss.item()))

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch+1}: "
            f"Train RMSE = {train_losses[-1]:.3f}, "
            f"Val RMSE = {val_losses[-1]:.3f}"
        )


### Continue baseline training to 1,000 epochs

In [ ]:

# Continue training the existing baseline model
for epoch in range(500):

    baseline_model.train()
    optimizer.zero_grad()

    predictions = baseline_model(X_train_tensor)
    loss = criterion(predictions, y_train_tensor)

    loss.backward()
    optimizer.step()

    # Evaluate after the update
    baseline_model.eval()
    with torch.no_grad():
        train_rmse = torch.sqrt(
            criterion(
                baseline_model(X_train_tensor),
                y_train_tensor
            )
        ).item()

        val_rmse = torch.sqrt(
            criterion(
                baseline_model(X_val_tensor),
                y_val_tensor
            )
        ).item()

    train_losses.append(train_rmse)
    val_losses.append(val_rmse)

    if (epoch + 1) % 100 == 0:
        print(
            f"Total Epoch {500 + epoch + 1}: "
            f"Train RMSE = {train_rmse:.3f}, "
            f"Val RMSE = {val_rmse:.3f}"
        )


In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(train_losses, label="Training RMSE")
plt.plot(val_losses, label="Validation RMSE")

plt.xlabel("Epoch")
plt.ylabel("RMSE (MPa)")
plt.title("Baseline ANN Learning Curves")
plt.legend()
plt.grid(True)
plt.show()


### Enhanced model: twelve features

In [ ]:

# Initialize our engineered ANN
torch.manual_seed(42)

engineered_model = ConcreteANN(input_dim=12)

criterion = nn.MSELoss()

optimizer_eng = torch.optim.Adam(
    engineered_model.parameters(),
    lr=0.001
)

eng_train_losses = []
eng_val_losses = []

for epoch in range(1000):

    # Training
    engineered_model.train()
    optimizer_eng.zero_grad()

    predictions = engineered_model(X_train_eng_tensor)
    loss = criterion(predictions, y_train_tensor)

    loss.backward()
    optimizer_eng.step()

    # Evaluation
    engineered_model.eval()

    with torch.no_grad():
        train_preds = engineered_model(X_train_eng_tensor)
        val_preds = engineered_model(X_val_eng_tensor)

        train_rmse = torch.sqrt(
            criterion(train_preds, y_train_tensor)
        ).item()

        val_rmse = torch.sqrt(
            criterion(val_preds, y_val_tensor)
        ).item()

    eng_train_losses.append(train_rmse)
    eng_val_losses.append(val_rmse)

    if (epoch + 1) % 100 == 0:
        print(
            f"Epoch {epoch+1}: "
            f"Train RMSE = {train_rmse:.3f}, "
            f"Val RMSE = {val_rmse:.3f}"
        )


In [ ]:

plt.figure(figsize=(9, 5))

plt.plot(val_losses,
         label="Baseline ANN (8 features)")

plt.plot(eng_val_losses,
         label="Engineered ANN (12 features)")

plt.xlabel("Epoch")
plt.ylabel("Validation RMSE (MPa)")
plt.title("Baseline vs Engineered ANN")
plt.legend()
plt.grid(True)
plt.show()


## 5. Five-fold group-aware cross-validation

A fresh scaler is fitted on each training fold and a new model is initialized for each fold. Identical input mixtures stay in one fold.

In [ ]:

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

def cross_validate_ann(X_data, y, groups,
                       epochs=1000, lr=0.001,
                       hidden1=32, hidden2=16,
                       weight_decay=0.0):

    kfold = GroupKFold(n_splits=5)
    results = []

    for fold, (tr_idx, va_idx) in enumerate(
        kfold.split(X_data, y, groups), start=1
    ):
        # Standardization inside each fold
        scaler = StandardScaler()

        X_tr = scaler.fit_transform(X_data.iloc[tr_idx])
        X_va = scaler.transform(X_data.iloc[va_idx])

        # Convert to PyTorch tensors
        X_tr = torch.tensor(X_tr, dtype=torch.float32)
        X_va = torch.tensor(X_va, dtype=torch.float32)

        y_tr = torch.tensor(
            y.iloc[tr_idx].to_numpy(),
            dtype=torch.float32
        ).reshape(-1, 1)

        y_va = torch.tensor(
            y.iloc[va_idx].to_numpy(),
            dtype=torch.float32
        ).reshape(-1, 1)

        # Create a new model for each fold
        torch.manual_seed(42 + fold)

        model = ConcreteANN(input_dim=X_data.shape[1], hidden1=hidden1, hidden2=hidden2)

        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        # Train the model
        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()

            pred = model(X_tr)
            loss = criterion(pred, y_tr)

            loss.backward()
            optimizer.step()

        # Evaluate the fold
        model.eval()
        with torch.no_grad():
            train_rmse = torch.sqrt(
                criterion(model(X_tr), y_tr)
            ).item()

            val_rmse = torch.sqrt(
                criterion(model(X_va), y_va)
            ).item()

        results.append({
            "Fold": fold,
            "Training RMSE": train_rmse,
            "Validation RMSE": val_rmse
        })

        print(
            f"Fold {fold}: "
            f"Train = {train_rmse:.3f}, "
            f"Validation = {val_rmse:.3f}"
        )

    return pd.DataFrame(results)


### Baseline versus engineered feature comparison

In [ ]:

# Baseline: 8 features
cv_baseline = cross_validate_ann(
    X, y, groups, epochs=1000
)

# Engineered: 12 features
cv_engineered = cross_validate_ann(
    X_engineered, y, groups, epochs=1000
)

# Compare mean validation RMSE
print("\nBaseline mean CV RMSE:",
      cv_baseline["Validation RMSE"].mean())

print("Engineered mean CV RMSE:",
      cv_engineered["Validation RMSE"].mean())

print("\nEngineered CV standard deviation:",
      cv_engineered["Validation RMSE"].std())


## 6. Hyperparameter experiments

Learning rate comparison (Adam).

In [ ]:

# Learning rate = 0.003
cv_lr003 = cross_validate_ann(
    X_engineered,
    y,
    groups,
    epochs=1000,
    lr=0.003
)

# Learning rate = 0.005
cv_lr005 = cross_validate_ann(
    X_engineered,
    y,
    groups,
    epochs=1000,
    lr=0.005
)

# Compare results
comparison = pd.DataFrame({
    "Learning Rate": [0.001, 0.003, 0.005],
    "Mean CV RMSE": [
        cv_engineered["Validation RMSE"].mean(),
        cv_lr003["Validation RMSE"].mean(),
        cv_lr005["Validation RMSE"].mean()
    ],
    "CV Standard Deviation": [
        cv_engineered["Validation RMSE"].std(),
        cv_lr003["Validation RMSE"].std(),
        cv_lr005["Validation RMSE"].std()
    ]
})

print(comparison)


### Hidden-layer architecture comparison

In [ ]:

# Architecture 64-32
cv_64_32 = cross_validate_ann(
    X_engineered,
    y,
    groups,
    epochs=1000,
    lr=0.003,
    hidden1=64,
    hidden2=32
)

# Architecture 128-64
cv_128_64 = cross_validate_ann(
    X_engineered,
    y,
    groups,
    epochs=1000,
    lr=0.003,
    hidden1=128,
    hidden2=64
)

# Compare results
architecture_results = pd.DataFrame({
    "Architecture": ["32-16", "64-32", "128-64"],
    "Mean CV RMSE": [
        cv_lr003["Validation RMSE"].mean(),
        cv_64_32["Validation RMSE"].mean(),
        cv_128_64["Validation RMSE"].mean()
    ],
    "CV Std": [
        cv_lr003["Validation RMSE"].std(),
        cv_64_32["Validation RMSE"].std(),
        cv_128_64["Validation RMSE"].std()
    ]
})

print(architecture_results)


### Weight-decay comparison

In [ ]:

# L2 regularization: weight decay = 0.0001
cv_wd0001 = cross_validate_ann(
    X_engineered, y, groups,
    epochs=1000,
    lr=0.003,
    hidden1=128,
    hidden2=64,
    weight_decay=0.0001
)

# L2 regularization: weight decay = 0.001
cv_wd001 = cross_validate_ann(
    X_engineered, y, groups,
    epochs=1000,
    lr=0.003,
    hidden1=128,
    hidden2=64,
    weight_decay=0.001
)

# Compare results
regularization_results = pd.DataFrame({
    "Weight Decay": [0.0, 0.0001, 0.001],
    "Mean Training RMSE": [
        cv_128_64["Training RMSE"].mean(),
        cv_wd0001["Training RMSE"].mean(),
        cv_wd001["Training RMSE"].mean()
    ],
    "Mean Validation RMSE": [
        cv_128_64["Validation RMSE"].mean(),
        cv_wd0001["Validation RMSE"].mean(),
        cv_wd001["Validation RMSE"].mean()
    ]
})

regularization_results


## 7. Early stopping

First, a holdout experiment illustrates the stopping criterion.

In [ ]:

torch.manual_seed(42)

model_es = ConcreteANN(
    input_dim=12,
    hidden1=128,
    hidden2=64
)

criterion = nn.MSELoss()

optimizer_es = torch.optim.Adam(
    model_es.parameters(),
    lr=0.003,
    weight_decay=0.0001
)

best_val_rmse = float("inf")
best_epoch = 0
best_weights = None

patience = 150
counter = 0

train_history = []
val_history = []

for epoch in range(3000):

    model_es.train()
    optimizer_es.zero_grad()

    pred = model_es(X_train_eng_tensor)
    loss = criterion(pred, y_train_tensor)

    loss.backward()
    optimizer_es.step()

    model_es.eval()
    with torch.no_grad():
        train_rmse = torch.sqrt(
            criterion(
                model_es(X_train_eng_tensor),
                y_train_tensor
            )
        ).item()

        val_rmse = torch.sqrt(
            criterion(
                model_es(X_val_eng_tensor),
                y_val_tensor
            )
        ).item()

    train_history.append(train_rmse)
    val_history.append(val_rmse)

    if val_rmse < best_val_rmse - 0.001:
        best_val_rmse = val_rmse
        best_epoch = epoch + 1
        best_weights = copy.deepcopy(model_es.state_dict())
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered")
        break

# Restore the best model
model_es.load_state_dict(best_weights)

print("Best epoch:", best_epoch)
print("Best validation RMSE:", best_val_rmse)


In [ ]:

plt.figure(figsize=(9, 5))

plt.plot(train_history, label="Training RMSE")
plt.plot(val_history, label="Validation RMSE")

plt.axvline(
    best_epoch,
    color="red",
    linestyle="--",
    label=f"Best epoch: {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("RMSE (MPa)")
plt.title("ANN Training with Early Stopping")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

# Evaluate the restored model
model_es.eval()

with torch.no_grad():
    restored_predictions = model_es(X_val_eng_tensor)

    restored_rmse = torch.sqrt(
        criterion(restored_predictions, y_val_tensor)
    ).item()

print(f"Best epoch: {best_epoch}")
print(f"Best validation RMSE: {best_val_rmse:.3f} MPa")
print(f"Restored model RMSE: {restored_rmse:.3f} MPa")
print(f"Total epochs performed: {len(val_history)}")


### Cross-validation with inner early stopping

Outer validation observations are excluded from stopping-epoch selection.

In [ ]:

def cross_validate_early_stopping(
    X_data, y, groups,
    max_epochs=3000,
    patience=150
):
    kfold = GroupKFold(n_splits=5)
    results = []

    for fold, (outer_tr, outer_val) in enumerate(
        kfold.split(X_data, y, groups), start=1
    ):

        # 1. Separate outer training and validation
        X_outer = X_data.iloc[outer_tr]
        y_outer = y.iloc[outer_tr]
        groups_outer = np.asarray(groups)[outer_tr]

        # 2. Create an inner split for early stopping
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=0.2,
            random_state=100 + fold
        )

        inner_tr, inner_val = next(
            splitter.split(X_outer, y_outer, groups_outer)
        )

        # 3. Standardize using inner training only
        inner_scaler = StandardScaler()

        X_tr = torch.tensor(
            inner_scaler.fit_transform(
                X_outer.iloc[inner_tr]
            ), dtype=torch.float32
        )

        X_stop = torch.tensor(
            inner_scaler.transform(
                X_outer.iloc[inner_val]
            ), dtype=torch.float32
        )

        y_tr = torch.tensor(
            y_outer.iloc[inner_tr].to_numpy(),
            dtype=torch.float32
        ).reshape(-1, 1)

        y_stop = torch.tensor(
            y_outer.iloc[inner_val].to_numpy(),
            dtype=torch.float32
        ).reshape(-1, 1)

        # 4. Train model with early stopping
        torch.manual_seed(42 + fold)

        model = ConcreteANN(
            input_dim=X_data.shape[1],
            hidden1=128,
            hidden2=64
        )

        criterion = nn.MSELoss()

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=0.003,
            weight_decay=0.0001
        )

        best_rmse = float("inf")
        best_epoch = 1
        counter = 0

        for epoch in range(max_epochs):
            model.train()
            optimizer.zero_grad()

            loss = criterion(model(X_tr), y_tr)
            loss.backward()
            optimizer.step()

            model.eval()
            with torch.no_grad():
                val_rmse = torch.sqrt(
                    criterion(model(X_stop), y_stop)
                ).item()

            if val_rmse < best_rmse - 0.001:
                best_rmse = val_rmse
                best_epoch = epoch + 1
                counter = 0
            else:
                counter += 1

            if counter >= patience:
                break

        # 5. Refit scaling on full outer training fold
        final_scaler = StandardScaler()

        X_full = torch.tensor(
            final_scaler.fit_transform(X_outer),
            dtype=torch.float32
        )

        X_test_fold = torch.tensor(
            final_scaler.transform(
                X_data.iloc[outer_val]
            ), dtype=torch.float32
        )

        y_full = torch.tensor(
            y_outer.to_numpy(), dtype=torch.float32
        ).reshape(-1, 1)

        y_test_fold = torch.tensor(
            y.iloc[outer_val].to_numpy(),
            dtype=torch.float32
        ).reshape(-1, 1)

        # 6. Retrain on full outer training fold
        torch.manual_seed(42 + fold)

        final_model = ConcreteANN(
            input_dim=X_data.shape[1],
            hidden1=128,
            hidden2=64
        )

        final_optimizer = torch.optim.Adam(
            final_model.parameters(),
            lr=0.003,
            weight_decay=0.0001
        )

        for epoch in range(best_epoch):
            final_model.train()
            final_optimizer.zero_grad()

            loss = criterion(
                final_model(X_full), y_full
            )
            loss.backward()
            final_optimizer.step()

        # 7. Evaluate on outer validation fold
        final_model.eval()

        with torch.no_grad():
            train_rmse = torch.sqrt(
                criterion(
                    final_model(X_full), y_full
                )
            ).item()

            outer_rmse = torch.sqrt(
                criterion(
                    final_model(X_test_fold),
                    y_test_fold
                )
            ).item()

        results.append({
            "Fold": fold,
            "Best Epoch": best_epoch,
            "Training RMSE": train_rmse,
            "Validation RMSE": outer_rmse
        })

        print(
            f"Fold {fold}: "
            f"Best Epoch = {best_epoch}, "
            f"Train = {train_rmse:.3f}, "
            f"Val = {outer_rmse:.3f}"
        )

    return pd.DataFrame(results)


In [ ]:

cv_early_stop = cross_validate_early_stopping(
    X_engineered,
    y,
    groups,
    max_epochs=3000,
    patience=150
)

print("\nCross-validation results:")
print(cv_early_stop)

print("\nMean validation RMSE:",
      cv_early_stop["Validation RMSE"].mean())

print("Validation RMSE standard deviation:",
      cv_early_stop["Validation RMSE"].std())

print("Mean best epoch:",
      cv_early_stop["Best Epoch"].mean())


## 8. Final model training and test-set evaluation

The selected model was retrained on the cleaned train/validation data. The test data was not used for parameter tuning.

In [ ]:

# Apply the same feature engineering to both datasets

def engineer_features(df):
    X_new = df.iloc[:, :-1].copy()

    cement = X_new.iloc[:, 0]
    slag = X_new.iloc[:, 1]
    fly_ash = X_new.iloc[:, 2]
    water = X_new.iloc[:, 3]
    age = X_new.iloc[:, 7]

    binder = cement + slag + fly_ash

    X_new["Total_Binder"] = binder
    X_new["Water_Binder_Ratio"] = water / binder
    X_new["SCM_Fraction"] = (slag + fly_ash) / binder
    X_new["Log_Age"] = np.log1p(age)

    return X_new

# Full training dataset (805 observations)
X_final = engineer_features(train_df)
y_final = train_df.iloc[:, -1]

# Supplied test dataset (206 observations)
X_test = engineer_features(test_df)
y_test = test_df.iloc[:, -1]

# Ensure features have identical column order
X_test = X_test[X_final.columns]

# Standardize using full training data only
final_scaler = StandardScaler()

X_final_scaled = final_scaler.fit_transform(X_final)
X_test_scaled = final_scaler.transform(X_test)

# Convert to PyTorch tensors
X_final_tensor = torch.tensor(
    X_final_scaled, dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_scaled, dtype=torch.float32
)

y_final_tensor = torch.tensor(
    y_final.to_numpy(), dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test.to_numpy(), dtype=torch.float32
).reshape(-1, 1)

print("Final training:", X_final_tensor.shape)
print("Final testing:", X_test_tensor.shape)


In [ ]:

# Reproducibility
torch.manual_seed(42)

# Initialize final neural network
final_model = ConcreteANN(
    input_dim=12,
    hidden1=128,
    hidden2=64
)

# Loss function
criterion = nn.MSELoss()

# Optimizer
final_optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=0.003,
    weight_decay=0.0001
)

# Training
final_train_losses = []

for epoch in range(1000):

    final_model.train()
    final_optimizer.zero_grad()

    predictions = final_model(X_final_tensor)

    loss = criterion(predictions, y_final_tensor)

    loss.backward()
    final_optimizer.step()

    # Record RMSE after the update
    final_model.eval()

    with torch.no_grad():
        train_rmse = torch.sqrt(
            criterion(
                final_model(X_final_tensor),
                y_final_tensor
            )
        ).item()

    final_train_losses.append(train_rmse)

    if (epoch + 1) % 100 == 0:
        print(
            f"Epoch {epoch+1}: "
            f"Training RMSE = {train_rmse:.3f} MPa"
        )


In [ ]:

# Switch to evaluation mode
final_model.eval()

with torch.no_grad():

    # Predict concrete compressive strengths
    test_predictions = final_model(X_test_tensor)

    # Calculate test RMSE
    test_rmse = torch.sqrt(
        criterion(test_predictions, y_test_tensor)
    ).item()

print(f"Final Test RMSE: {test_rmse:.3f} MPa")


### Actual versus predicted concrete strength

In [ ]:

actual = y_test_tensor.numpy().flatten()
predicted = test_predictions.numpy().flatten()

plt.figure(figsize=(7, 6))

plt.scatter(actual, predicted, alpha=0.6)

limits = [
    min(actual.min(), predicted.min()),
    max(actual.max(), predicted.max())
]

plt.plot(
    limits, limits,
    'r--',
    label="Perfect prediction"
)

plt.xlabel("Actual Compressive Strength (MPa)")
plt.ylabel("Predicted Compressive Strength (MPa)")
plt.title("Actual vs Predicted Concrete Strength")

plt.legend()
plt.grid(True)
plt.show()


### Sensitivity check: unseen input mixtures

In [ ]:

# Identify genuinely unseen concrete mixtures
feature_cols = train_df.columns[:-1]

train_keys = pd.MultiIndex.from_frame(
    train_df[feature_cols]
)

test_keys = pd.MultiIndex.from_frame(
    test_df[feature_cols]
)

unseen_mask = ~test_keys.isin(train_keys)

# Calculate RMSE on unseen mixtures only
mask_tensor = torch.tensor(unseen_mask, dtype=torch.bool)

with torch.no_grad():
    unseen_rmse = torch.sqrt(
        torch.mean(
            (test_predictions[mask_tensor]
             - y_test_tensor[mask_tensor]) ** 2
        )
    ).item()

print("Unseen test observations:", unseen_mask.sum())
print(f"Unseen-mixture RMSE: {unseen_rmse:.3f} MPa")


### Residual analysis

In [ ]:

# Actual and predicted compressive strengths
actual = y_test_tensor.detach().cpu().numpy().flatten()
predicted = test_predictions.detach().cpu().numpy().flatten()

# Calculate residuals
residuals = actual - predicted

plt.figure(figsize=(9, 5))

plt.scatter(
    predicted,
    residuals,
    alpha=0.65,
    edgecolor="black",
    linewidth=0.3
)

# Zero-residual reference line
plt.axhline(
    y=0,
    color="red",
    linestyle="--",
    label="Zero residual"
)

plt.xlabel("Predicted Compressive Strength (MPa)")
plt.ylabel("Residual (Actual - Predicted) (MPa)")
plt.title("Residual Plot for Final PyTorch ANN")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("Residual_Plot.png", dpi=300)
plt.show()

print(f"Mean residual: {residuals.mean():.3f} MPa")
print(f"Residual standard deviation: {residuals.std():.3f} MPa")


## 9. SHAP model interpretation

SHAP explains the model output relative to a reference sample, not independent causal effects. Engineered inputs are correlated.

In [ ]:

# Ensure model is in evaluation mode
final_model.eval()

rng = np.random.default_rng(42)

# Select background observations from training data
background_idx = rng.choice(
    len(X_final_tensor),
    size=100,
    replace=False
)

background = X_final_tensor[background_idx]

# Select previously unseen test mixtures
unseen_indices = np.flatnonzero(unseen_mask)

sample_indices = rng.choice(
    unseen_indices,
    size=min(100, len(unseen_indices)),
    replace=False
)

X_explain = X_test_tensor[sample_indices]

# Initialize Deep SHAP
explainer = shap.DeepExplainer(
    final_model,
    background
)

# Calculate SHAP values
shap_values = explainer.shap_values(X_explain)

# Handle different SHAP output formats
if isinstance(shap_values, list):
    shap_values = shap_values[0]

shap_values = np.asarray(shap_values)

if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 0]

print("SHAP values shape:", shap_values.shape)


In [ ]:

# Reference model prediction
base_value = float(
    np.asarray(explainer.expected_value).reshape(-1)[0]
)

# Package SHAP results
shap_exp = shap.Explanation(
    values=shap_values,
    base_values=np.full(len(sample_indices), base_value),
    data=X_test.iloc[sample_indices].to_numpy(),
    feature_names=list(X_final.columns)
)


In [ ]:

shap.plots.bar(
    shap_exp,
    max_display=12,
    show=False
)

plt.title("SHAP Feature Importance - PyTorch ANN")
plt.tight_layout()
plt.savefig("SHAP_Feature_Importance.png",
            dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

shap.plots.beeswarm(
    shap_exp,
    max_display=12,
    show=False
)

plt.title("SHAP Summary - Concrete Strength ANN")
plt.tight_layout()
plt.savefig("SHAP_Beeswarm.png",
            dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

# Explain the first selected concrete mixture
shap.plots.waterfall(
    shap_exp[0],
    max_display=12,
    show=False
)

plt.tight_layout()
plt.savefig("SHAP_Waterfall.png",
            dpi=300, bbox_inches="tight")
plt.show()


## 10. Interpretation and limitations

- The selected configuration was 12 inputs → 128 → 64 → 1, ReLU, Adam (learning rate 0.003; weight decay 0.0001; 1,000 epochs).
- Reference results from the original run: **5-fold CV RMSE 5.116 MPa**, **test RMSE 4.418 MPa**, **unseen-mixture test RMSE 4.390 MPa**.
- Dataset overlap: 10 of the test mixtures match an input combination in the cleaned train/validation data. Report both full test and unseen-mixture results.
- SHAP feature contributions are affected by correlated engineered inputs. Do not equate attribution with causation.
- The test split is held out during model selection. Further tuning on the test set would compromise its independence.
- Results may vary slightly across runtime/library versions and numerical backends.

The figures generated by the original run are available under `figures/` for review without executing the full notebook.

In [ ]:
# SHAP additivity verification (run after SHAP computation)
final_model.eval()
with torch.no_grad():
    reconstructed_targets = final_model(X_explain).cpu().numpy().ravel()
recon = base_value + shap_values.sum(axis=1)
print('Maximum absolute SHAP reconstruction error:', np.abs(recon - reconstructed_targets).max())
